# POLISY lab

Runs the POLISY pipeline (the modules in `POLISY_DA.zip`) from top to bottom.

**Before you start:** upload `POLISY_DA.zip` and your downloads to the Colab file panel (`/content`), or keep the
downloads in a Google Drive folder and add that folder in step 2. File names do not have to match: every input is
found by name *and* by what is inside it, and step 01 prints which file was used for each one.

**Keep the code in the zip.** Do not paste module code into cells or edit `polisy_core` in a cell: the cell and the
`.py` file drift apart. That drift is how `CBSA_REFERENCE` ended up pointing at three different places.

When a file is MISSING or AMBIGUOUS, see the file-name checklist: https://claude.ai/code/artifact/1aa6867a-4183-4491-a34a-c7f41a86be48

In [ ]:
%pip install -q duckdb rapidfuzz

In [ ]:
# 1. Load the pipeline code from the newest POLISY_DA*.zip in /content, and only from there.
import sys, zipfile
from pathlib import Path

CODE = Path("/content/polisy_code")
zips = sorted(Path("/content").glob("POLISY_DA*.zip"), key=lambda p: p.stat().st_mtime)
if zips:
    with zipfile.ZipFile(zips[-1]) as zf:
        zf.extractall(CODE)
core = (next(CODE.rglob("polisy_core.py"), None) if CODE.exists() else None) or next(Path("/content").glob("polisy_core.py"), None)
if core is None:
    raise FileNotFoundError("Upload POLISY_DA.zip (or the .py files) to /content first.")
sys.path.insert(0, str(core.parent))
sys.modules.pop("polisy_core", None)          # never keep an older copy loaded
import polisy_core as pc
print("polisy_core", pc.__version__, "loaded from", pc.__file__)

In [ ]:
# 2. Only if needed; skip otherwise.
# Downloads in Google Drive? Mount it and add the folder to the search:
# from google.colab import drive; drive.mount("/content/drive")
# pc.CONFIG["SEARCH_DIRS"].insert(0, "/content/drive/MyDrive/POLISY")
#
# Step 01 used the wrong file for an input, or says AMBIGUOUS / MISSING? Point that input at the exact file:
# pc.CONFIG["DIPI"] = "/content/Organizational Leadership File.csv"
# pc.CONFIG["CBSA_REFERENCE"] = "/content/list1_2023 (1).xlsx"
# pc.CONFIG["OEWS_MSA"] = {2024: "/content/oesm24ma (1).zip"}
#
# Other settings: pc.CONFIG["OEWS_YEARS"], pc.CONFIG["DIPI_MEASURE"], pc.CONFIG["CENSUS_API_KEY"]

## 05 Downloads: O*NET, AIOE, Census delineation file, ACS

Run this first: module 04 needs the Census file (List 1) and 06 the O*NET and AIOE files. Files you already have, under any name, are skipped. The OEWS zips are never downloaded here; get them by hand from bls.gov.

In [ ]:
pc.run("05")

## 01 Inventory: which file is used for each input

Every input should say `ok`. Read the `found_by` column: `CONFIG path` and `download name` need no action; `similar name` and `file contents` are fine but deserve one look at the file column. DIPI and the CBSA reference are read end to end here, so a file that is present but unusable fails now, not in 04 or 07.

In [ ]:
files = pc.run("01")
files

## 02 VRscores to Parquet

The four panels all download as `dataverse_files.zip`; they are told apart by the files inside.

In [ ]:
pc.run("02")

## 03 BLS OEWS

Needs oesm24nat.zip, oesm24ma.zip and oesm24in4.zip (or unzipped copies) in `/content` or `data/raw`.

In [ ]:
pc.run("03")

## 04 Crosswalks: CBSA reference, occupations, industries, firm name match

The log line `CBSA reference: <file>, N CBSAs, M counties` means the Census file was used; `OEWS metro file` means it was not found and the election merge in 06 will be skipped.

In [ ]:
pc.run("04", year=2015)

## 06 Merging

In [ ]:
pc.run("06")

## 07 Validation (includes the DIPI check)

In [ ]:
pc.run("07")

## 08 Descriptives

In [ ]:
pc.run("08")

## 09 Question 1: partisan asymmetry

In [ ]:
pc.run("09")

## 10 Question 2: occupations and places

In [ ]:
pc.run("10")